In [2]:
import pandas as pd
import numpy as np
from pathlib import Path

# Update these paths if your files are stored elsewhere.
CITY_DAY_PATH = Path('/content/city_day.csv')
AQI_PATH = Path('/content/aqi.csv')
OUTPUT_PATH = Path('/content/master_aqi_pipeline.csv')

if not CITY_DAY_PATH.exists() or not AQI_PATH.exists():
    raise FileNotFoundError(
        "Could not find city_day.csv and/or aqi.csv. Upload both files to /content "
        "or change CITY_DAY_PATH and AQI_PATH to the correct locations."
    )

df_2015 = pd.read_csv(CITY_DAY_PATH)
df_2023 = pd.read_csv(AQI_PATH)

df_2023 = df_2023.rename(columns={
    'area': 'City', 'date': 'Date', 'aqi_value': 'AQI',
    'air_quality_status': 'AQI_Bucket'
})

required = {'City', 'Date', 'AQI'}
for name, frame in [('city_day.csv', df_2015), ('aqi.csv', df_2023)]:
    missing = required - set(frame.columns)
    if missing:
        raise ValueError(f"{name} is missing required columns: {sorted(missing)}")

for frame in (df_2015, df_2023):
    frame['City'] = frame['City'].astype('string').str.strip().str.title()
    frame['Date'] = pd.to_datetime(frame['Date'], errors='coerce', dayfirst=True)
    frame['AQI'] = pd.to_numeric(frame['AQI'], errors='coerce')

master_df = pd.concat([df_2015, df_2023], ignore_index=True, sort=False)
master_df = master_df.dropna(subset=['City', 'Date', 'AQI'])
master_df = master_df[master_df['City'].str.len() > 0]
master_df = master_df.drop_duplicates(subset=['City', 'Date'], keep='last')
master_df = master_df.sort_values(['City', 'Date']).reset_index(drop=True)
master_df.to_csv(OUTPUT_PATH, index=False)

print(f"Loaded {len(master_df):,} city-date records across {master_df['City'].nunique()} cities.")
print('Cities available:', ', '.join(sorted(master_df['City'].unique())))

Loaded 12,237 city-date records across 257 cities.
Cities available: Agartala, Agra, Ahmedabad, Ahmednagar, Aizawl, Ajmer, Akola, Alwar, Amaravati, Amravati, Amritsar, Anantapur, Angul, Ankleshwar, Araria, Ariyalur, Arrah, Asansol, Aurangabad, Baddi, Badlapur, Bagalkot, Baghpat, Balasore, Banswara, Baran, Barbil, Bareilly, Baripada, Barmer, Barrackpore, Bathinda, Begusarai, Belapur, Belgaum, Bengaluru, Bettiah, Bhagalpur, Bharatpur, Bhilai, Bhilwara, Bhiwadi, Bhiwandi, Bhopal, Bhubaneswar, Bidar, Bihar Sharif, Bikaner, Bilaspur, Bileipada, Boisar, Brajrajnagar, Bulandshahr, Bundi, Buxar, Byasanagar, Byrnihat, Chamarajanagar, Chandigarh, Chandrapur, Charkhi Dadri, Chengalpattu, Chennai, Chhal, Chhapra, Chikkaballapur, Chikkamagaluru, Chittoor, Chittorgarh, Churu, Coimbatore, Cuddalore, Cuttack, Damoh, Dausa, Davanagere, Dehradun, Delhi, Dewas, Dhanbad, Dharwad, Dholpur, Dhule, Dindigul, Dungarpur, Durgapur, Eloor, Faridabad, Firozabad, Gadag, Gandhi Nagar, Gangtok, Gaya, Ghaziabad, Gora

In [4]:
# Create calendar features for every city (no city is hard-coded).
df = pd.read_csv(OUTPUT_PATH, parse_dates=['Date'])
df['Year'] = df['Date'].dt.year
df['Month'] = df['Date'].dt.month
df['Day'] = df['Date'].dt.day
df['DayOfYear'] = df['Date'].dt.dayofyear
df['Weekday'] = df['Date'].dt.dayofweek
df['Season'] = df['Month'].map({12:'Winter', 1:'Winter', 2:'Spring', 3:'Spring',
                                4:'Summer', 5:'Summer', 6:'Summer', 7:'Summer',
                                8:'Monsoon', 9:'Monsoon', 10:'Autumn', 11:'Autumn'})
df['Is_Lohri_Month'] = (df['Month'] == 1).astype(int)
df['Is_Diwali_Season'] = df['Month'].isin([10, 11]).astype(int)
df['Is_Paddy_Harvesting'] = (
    df['Month'].isin([10, 11]) | ((df['Month'] == 9) & (df['Day'] >= 15))
).astype(int)

# Keep only records with target AQI and valid city/date.
df = df.dropna(subset=['City', 'Date', 'AQI']).copy()
print('Rows:', len(df), '| Cities:', df['City'].nunique())
display(df[['City', 'Date', 'AQI', 'Year', 'Month', 'Day', 'Season']].head())

Rows: 12237 | Cities: 257


,City,Date,AQI,Year,Month,Day,Season
0,Agartala,2025-03-25,225.0,2025,3,25,Spring
1,Agartala,2025-03-26,299.0,2025,3,26,Spring
2,Agartala,2025-03-27,89.0,2025,3,27,Spring
3,Agartala,2025-03-28,40.0,2025,3,28,Spring
4,Agartala,2025-03-29,40.0,2025,3,29,Spring


In [5]:
# Train a city-aware model. OneHotEncoder handles city names without assigning
# arbitrary numeric order to cities. Date features let the model learn broad trends.
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import joblib

FEATURES = ['City', 'Year', 'Month', 'Day', 'DayOfYear', 'Weekday',
            'Is_Lohri_Month', 'Is_Diwali_Season', 'Is_Paddy_Harvesting']
TARGET = 'AQI'
model_df = df[FEATURES + [TARGET]].copy()
model_df = model_df.replace([np.inf, -np.inf], np.nan).dropna(subset=[TARGET, 'City'])
X = model_df[FEATURES]
y = model_df[TARGET]

# Random split for a basic evaluation. For formal forecasting evaluation, use a
# chronological split so future dates are held out from training.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
preprocessor = ColumnTransformer(transformers=[
    ('city', OneHotEncoder(handle_unknown='ignore'), ['City']),
    ('numeric', Pipeline([('imputer', SimpleImputer(strategy='median'))]),
     [c for c in FEATURES if c != 'City'])
])
model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', RandomForestRegressor(n_estimators=300, random_state=42, min_samples_leaf=2))
])
model.fit(X_train, y_train)
pred = model.predict(X_test)
print('MAE:', round(mean_absolute_error(y_test, pred), 2))
print('RMSE:', round(np.sqrt(mean_squared_error(y_test, pred)), 2))
print('R²:', round(r2_score(y_test, pred), 3))

MODEL_PATH = '/content/city_aqi_model.joblib'
joblib.dump({'model': model, 'features': FEATURES, 'cities': sorted(df['City'].unique())}, MODEL_PATH)
print('Saved model to', MODEL_PATH)

MAE: 28.85
RMSE: 57.62
R²: 0.72
Saved model to /content/city_aqi_model.joblib


In [6]:
# USER INPUT: select a city and date, then predict AQI.
from datetime import date

available_cities = sorted(df['City'].dropna().unique().tolist())

def make_prediction(city, prediction_date):
    city = str(city).strip().title()
    prediction_date = pd.to_datetime(prediction_date, errors='raise')
    if city not in available_cities:
        raise ValueError(f"'{city}' is not in the training data. Choose one of: {available_cities}")

    month = prediction_date.month
    row = pd.DataFrame([{
        'City': city,
        'Year': prediction_date.year,
        'Month': month,
        'Day': prediction_date.day,
        'DayOfYear': prediction_date.dayofyear,
        'Weekday': prediction_date.dayofweek,
        'Is_Lohri_Month': int(month == 1),
        'Is_Diwali_Season': int(month in [10, 11]),
        'Is_Paddy_Harvesting': int(month in [10, 11] or (month == 9 and prediction_date.day >= 15))
    }], columns=FEATURES)
    estimate = float(model.predict(row)[0])
    return max(0, estimate)

# Preferred interactive dropdown in Jupyter/Colab; falls back to text input if
# ipywidgets is unavailable.
try:
    import ipywidgets as widgets
    from IPython.display import display, clear_output

    city_dropdown = widgets.Dropdown(options=available_cities, description='City:',
                                     value='Chandigarh' if 'Chandigarh' in available_cities else available_cities[0],
                                     layout=widgets.Layout(width='350px'))
    date_picker = widgets.DatePicker(description='Prediction date:', value=date.today())
    predict_button = widgets.Button(description='Predict AQI', button_style='primary')
    output = widgets.Output()

    def on_predict_clicked(_):
        with output:
            clear_output()
            if date_picker.value is None:
                print('Please select a date.')
                return
            estimate = make_prediction(city_dropdown.value, date_picker.value)
            print(f'City: {city_dropdown.value}')
            print(f'Date: {date_picker.value}')
            print(f'Predicted AQI: {estimate:.1f}')
            print('Note: this is a city/date-based estimate, not a live AQI reading.')

    predict_button.on_click(on_predict_clicked)
    display(widgets.VBox([city_dropdown, date_picker, predict_button, output]))
except ImportError:
    print('ipywidgets is unavailable; enter values below.')
    city_input = input(f"City ({', '.join(available_cities)}): ")
    date_input = input('Date (YYYY-MM-DD): ')
    print(f'Predicted AQI: {make_prediction(city_input, date_input):.1f}')